# Task 1 Crawling

This notebook builds a corpus of smartphone-related user opinions from Hacker News using the `Algolia API` and the official `HN Firebase API`. 

The devices are: 
- iPhone 15, iPhone 15 Pro, iPhone 15 Pro Max, iPhone 14, iPhone 14 Pro, iPhone 14 Pro Max, iPhone 13, iPhone 13 Pro, iPhone 13 Pro Max, Galaxy S24, Galaxy S23, Galaxy S22, Galaxy S21, Pixel 8, Pixel 7, Pixel 6, OnePlus 12, OnePlus 13, Xiaomi 14, Xiaomi 14 Ultra, Xiaomi 15, Xiaomi 15 Ultra, Galaxy Z Fold5, Galaxy Z Flip5, Galaxy S24 Ultra, Galaxy S23 Ultra, Galaxy S22 Ultra, Galaxy S21 Ultra, iPhone 17, iPhone 17 Pro, and iPhone 17 Pro Max

The workflow is as follows:
 1. Crawls Hacker News stories and comments matching device-related queries (e.g. "Pixel 8 battery", "Galaxy S23 review") for the selected smartphones via Algolia API, then enriches results with the official Firebase API (deleted, descendants, score, item_type).
 2. Processes the raw data by removing HTML, deduplicating entries, applying quality filters, and sanitizing text for compatibility with Excel exports.
 3. Saves results to `corpus_full.xlsx` (containing all metadata) and `eval.xlsx` (containing text and annotation label columns).

## 1. Importing libraries

In [2]:
# !pip -q install pandas requests openpyxl

In [3]:
import html
import os
import re
import time
import json
import random
import hashlib
import unicodedata
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone

## 2. Configuration

In [ ]:
# Target devices to search for in HN comments and stories
DEVICES = [
    "iPhone 15",
    "iPhone 15 Pro",
    "iPhone 15 Pro Max",
    "iPhone 14",
    "iPhone 14 Pro",
    "iPhone 14 Pro Max",
    "iPhone 13",
    "iPhone 13 Pro",
    "iPhone 13 Pro Max",
    "Galaxy S24",
    "Galaxy S23",
    "Galaxy S22",
    "Galaxy S21",
    "Pixel 8",
    "Pixel 7",
    "Pixel 6",
    "OnePlus 12",
    "Xiaomi 14"
]

ALGOLIA_SEARCH_BY_DATE = "https://hn.algolia.com/api/v1/search_by_date"
HN_FIREBASE_BASE = "https://hacker-news.firebaseio.com/v0"
FIREBASE_SLEEP = 0.15

## 3. Crawling & initial processing

- We crawl Hacker News via the Algolia API using time-sliced queries. For each device, we search for "device name", "review", "battery", "camera", "problems", "worth it" etc as defined in `build_hn_queries()` below. We also add comparisons like "iPhone 15 vs Galaxy S24".

- The same HN post can match multiple queries (e.g. a comment about "iPhone 15 battery" appears in both "iPhone 15" and "iPhone 15 battery"). Therefore, we assign each document a unique `doc_id` and deduplicate to keep one row per unique post.

- After crawling, we further enrich the results by using the official HN Firebase API (`/v0/item/{id}.json`) to add attributes like:
     - `deleted` (whether the post/comment was deleted)
     - `descendants` (number of direct replies for comments, or total descendants for stories)
     - `score` (number of upvotes; for stories only, comments have 0)

### Crawl utilities

In [ ]:
# Normalize and clean up a string
def normalize_text(s):
    s = (s or "").strip()
    s = re.sub(r"\s+", " ", s)  # squeeze spaces
    return s

def word_count(s):
    return len((s or "").split())

# Generate a unique doc_id from document source, id, and text snippet
def make_doc_id(source, object_id, text):
    base = f"{source}|{object_id}|{(text or '')[:500]}"
    return hashlib.sha1(base.encode("utf-8", errors="ignore")).hexdigest()

# Remove HTML tags and unescape entities from a string
def clean_html():
    if not s:
        return ""
    s = re.sub(r"<.*?>", " ", s)
    s = html.unescape(s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

# Normalize text for phrase matching
def normalize_for_match(s):
    s = (s or "").lower()
    s = html.unescape(s)
    s = re.sub(r"<.*?>", " ", s)
    s = re.sub(r"[^a-z0-9+ ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Get search aliases for a device e.g., "iPhone 15" -> ["iphone 15", "ip 15"]
def get_device_aliases(device):
    d = normalize_for_match(device)
    aliases = [d]
    if d.startswith("iphone "):
        rest = d[7:]
        aliases.append(f"ip {rest}")
    return aliases

# Check if text contains device name
def contains_device_phrase(text, device):
    text_n = normalize_for_match(text)
    device_n = normalize_for_match(device)

    # prevent iPhone 15 from incorrectly matching iPhone 15 Pro / Pro Max
    if device_n == "iphone 15":
        if re.search(r"\b(?:iphone\s*15|ip\s*15)\s*pro\b", text_n):
            return False
    if device_n == "iphone 14":
        if re.search(r"\b(?:iphone\s*14|ip\s*14)\s*pro\b", text_n):
            return False
    if device_n == "iphone 13":
        if re.search(r"\b(?:iphone\s*13|ip\s*13)\s*pro\b", text_n):
            return False

    # Match any alias and allow no space e.g., iphone15, ip15
    for alias in get_device_aliases(device):
        pattern = r"\b" + re.escape(alias).replace(r"\ ", r"\s*") + r"\b"
        if re.search(pattern, text_n):
            return True
    return False

### Algolia API functions

In [ ]:
# Convert a datetime object to a unix timestamp
def dt_to_unix(dt):
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return int(dt.timestamp())

# Query Algolia API
def algolia_get(url, params, retries=5, headers=None):
    hdrs = headers or {"User-Agent": "Mozilla/5.0 (SC4021-opinion-crawler/1.0)"}
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, headers=hdrs, timeout=25)
            r.raise_for_status()
            return r.json()
        except (requests.HTTPError, requests.RequestException) as e:
            sleep_s = (2 ** attempt) + random.random()
            print("Req error:", e, "| sleeping", round(sleep_s, 2), "s")
            time.sleep(sleep_s)
    return None

def build_hn_queries(devices):
    templates = [
        '"{d}"',
        '"{d}" review',
        '"{d}" "long term review"',
        '"{d}" "worth it"',
        '"{d}" "should i buy"',
        '"{d}" upgrade',
        '"{d}" problems',
        '"{d}" issues',
        '"{d}" battery',
        '"{d}" camera',
        '"{d}" overheating',
        '"{d}" performance',
        '"{d}" regret',
        '"{d}" love',
        '"{d}" hate',
        '"{d}" "first impressions"',
        '"{d}" recommend',
        '"{d}" opinion',
        '"{d}" "switched from"',
        '"{d}" "compared to"'
    ]

    pairs = []
    seen = set()

    for d in devices:
        # Query with full device name
        for t in templates:
            q = t.format(d=d).strip()
            key = (d, q)
            if key not in seen:
                pairs.append((d, q))
                seen.add(key)
        # For iPhone, also query with "ip" alias 
        if d.lower().startswith("iphone "):
            ip_alias = "ip " + d[7:]
            for t in templates:
                q = t.format(d=ip_alias).strip()
                key = (d, q)
                if key not in seen:
                    pairs.append((d, q))
                    seen.add(key)

    return pairs

# Crawl Hacker News via Algolia API with time-sliced queries
def crawl_hn_time_sliced(query_pairs, start_date, end_date, slice_days=30, hits_per_page=200,
    max_pages_per_slice=5, tags="(comment,story)", min_chars=80, sleep_s=0.5,
    checkpoint_path=None, resume=False, apply_device_filter=True):
    rows = []
    start_idx = 0
    start_date = start_date.replace(tzinfo=timezone.utc)
    end_date = end_date.replace(tzinfo=timezone.utc)

    # Resume from checkpoint if requested
    if checkpoint_path and resume and os.path.exists(checkpoint_path):
        state_path = checkpoint_path.replace(".parquet", "_state.json")
        if os.path.exists(state_path):
            df_check = pd.read_parquet(checkpoint_path)
            rows = df_check.to_dict("records")
            with open(state_path, "r") as f:
                state = json.load(f)
            start_idx = state.get("last_query_idx", -1) + 1
            print(f"Resuming from query pair {start_idx + 1}/{len(query_pairs)} ({len(rows)} rows so far)")

    for qi, (device, q) in enumerate(query_pairs, start=1):
        if qi - 1 < start_idx:
            continue
        print(f"\n[{qi}/{len(query_pairs)}] Device: {device} | HN query: {q}")
        win_start = start_date

        while win_start < end_date:
            win_end = min(win_start + timedelta(days=slice_days), end_date)
            a, b = dt_to_unix(win_start), dt_to_unix(win_end)
            nf = [f"created_at_i>{a}", f"created_at_i<{b}"]

            for page in range(max_pages_per_slice):
                params = {
                    "query": q,
                    "tags": tags,
                    "numericFilters": ",".join(nf),
                    "hitsPerPage": hits_per_page,
                    "page": page
                }

                data = algolia_get(ALGOLIA_SEARCH_BY_DATE, params=params)
                if not data:
                    break

                hits = data.get("hits", []) or []
                if not hits:
                    break

                for h in hits:
                    title_raw = h.get("title") or ""
                    story_title_raw = h.get("story_title") or ""
                    body_raw = h.get("comment_text") or h.get("story_text") or ""

                    title_clean = clean_html(title_raw)
                    story_title_clean = clean_html(story_title_raw)
                    body_clean = clean_html(body_raw)

                    combined_text = " ".join(
                        x for x in [title_clean, story_title_clean, body_clean] if x
                    ).strip()

                    if len(combined_text) < min_chars:
                        continue

                    if apply_device_filter and not contains_device_phrase(combined_text, device):
                        continue

                    obj_id = str(h.get("objectID", ""))
                    url = h.get("url") or h.get("story_url") or ""
                    created = h.get("created_at") or ""
                    author = h.get("author") or ""
                    ttags = ",".join(h.get("_tags", [])) if isinstance(h.get("_tags"), list) else ""

                    rows.append({
                        "doc_id": make_doc_id("hn_algolia", obj_id, combined_text),
                        "object_id": obj_id,
                        "device": device,
                        "title": normalize_text(title_clean or story_title_clean),
                        "query": q,
                        "created_at": created,
                        "author": author,
                        "tags": ttags,
                        "source_url": url,
                        "text": normalize_text(combined_text)
                    })

                time.sleep(sleep_s + random.random() * 0.3)

                if page + 1 >= data.get("nbPages", 0):
                    break

            win_start = win_end

        # Save after each query pair
        if checkpoint_path and rows:
            df_check = pd.DataFrame(rows)
            df_check.to_parquet(checkpoint_path, index=False)
            state_path = checkpoint_path.replace(".parquet", "_state.json")
            with open(state_path, "w") as f:
                json.dump({"last_query_idx": qi - 1, "total_rows": len(rows)}, f)

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.drop_duplicates(subset=["doc_id"])
        df = df.drop_duplicates(subset=["text"])
        df = df[df["text"].astype(str).str.len() > 0]
    return df


### Firebase API functions

In [ ]:
# Get an item's JSON from the HN Firebase API
def hn_firebase_get(item_id, retries=5):
    url = f"{HN_FIREBASE_BASE}/item/{item_id}.json"
    for attempt in range(retries):
        try:
            r = requests.get(url, headers={"User-Agent": "SC4021-opinion-crawler/1.0"}, timeout=15)
            # Handle rate limiting by waiting and retrying
            if r.status_code == 429:
                wait = (2 ** attempt) + random.uniform(1, 3)
                print(f"  Rate limit (429) | waiting {wait:.1f}s")
                time.sleep(wait)
                continue
            r.raise_for_status()
            data = r.json()
            if data is None:
                return {"deleted": True, "descendants": 0}
            return data
        except requests.RequestException as e:
            wait = (2 ** attempt) + random.uniform(0.5, 2)
            if attempt < retries - 1:
                print(f"  Firebase error: {e} | retry in {wait:.1f}s")
                time.sleep(wait)
            else:
                return None
    return None

# Enrich the corpus with Firebase API data
def enrich_with_firebase(df, wait=0.15):
    if "object_id" not in df.columns:
        return df
    out = df.copy()
    out["deleted"] = False
    out["descendants"] = 0
    out["score"] = 0
    out["item_type"] = ""
    seen = {}
    ids = [x for x in out["object_id"].unique().tolist() if x is not None and str(x).strip()]
    n = len(ids)

    # Iterate through each unique object_id to fetch data from Firebase
    for i, oid in enumerate(ids):
        if (i + 1) % 500 == 0 or i == 0:
            print(f"  [{i+1}/{n}]")
        if (i + 1) % 1000 == 0 and i > 0:
            time.sleep(5)

        item = hn_firebase_get(oid)
        if item is None:
            seen[oid] = {"deleted": False, "descendants": 0, "score": 0, "item_type": ""}
        else:
            deleted = item.get("deleted") or item.get("dead") or False
            desc = item.get("descendants")  # Get number of comments
            if desc is None and "kids" in item:  # Fallback to counting 'kids'
                desc = len(item["kids"]) if isinstance(item["kids"], list) else 0
            else:
                desc = int(desc) if desc is not None else 0
            score = int(item.get("score") or 0)
            item_type = item.get("type") or ""
            seen[oid] = {"deleted": deleted, "descendants": desc, "score": score, "item_type": item_type}
        time.sleep(wait)

    # Enrich the df with fetched fields
    for idx, oid in out["object_id"].items():
        v = seen.get(oid, {"deleted": False, "descendants": 0, "score": 0, "item_type": ""})
        out.at[idx, "deleted"] = v["deleted"]
        out.at[idx, "descendants"] = v["descendants"]
        out.at[idx, "score"] = v["score"]
        out.at[idx, "item_type"] = v["item_type"]
    n_del = out["deleted"].sum()
    if n_del > 0:
        print(f"  Found {int(n_del)} deleted items")
    return out

### Data crawling from Algolia API

In [ ]:
hn_queries = build_hn_queries(DEVICES)  # returns list of (device, query) tuples
start = datetime(2021, 1, 1, tzinfo=timezone.utc)
end = datetime.now(timezone.utc)

# Saves after each query pair
CHECKPOINT_PATH = "hn_crawl_checkpoint.parquet"

df_hn = crawl_hn_time_sliced(
    query_pairs=hn_queries,
    start_date=start,
    end_date=end,
    slice_days=14,
    hits_per_page=500,
    max_pages_per_slice=10,
    tags="(comment,story)",
    min_chars=60,
    checkpoint_path=CHECKPOINT_PATH,
    resume=False,
    apply_device_filter=False
)

print("HN rows:", len(df_hn), "| words:", int(df_hn["text"].map(word_count).sum()))


[1/540] Device: iPhone 15 | HN query: "iPhone 15"

[2/540] Device: iPhone 15 | HN query: "iPhone 15" review

[3/540] Device: iPhone 15 | HN query: "iPhone 15" "long term review"

[4/540] Device: iPhone 15 | HN query: "iPhone 15" "worth it"

[5/540] Device: iPhone 15 | HN query: "iPhone 15" "should i buy"

[6/540] Device: iPhone 15 | HN query: "iPhone 15" upgrade

[7/540] Device: iPhone 15 | HN query: "iPhone 15" problems

[8/540] Device: iPhone 15 | HN query: "iPhone 15" issues

[9/540] Device: iPhone 15 | HN query: "iPhone 15" battery

[10/540] Device: iPhone 15 | HN query: "iPhone 15" camera

[11/540] Device: iPhone 15 | HN query: "iPhone 15" overheating

[12/540] Device: iPhone 15 | HN query: "iPhone 15" performance

[13/540] Device: iPhone 15 | HN query: "iPhone 15" regret

[14/540] Device: iPhone 15 | HN query: "iPhone 15" love

[15/540] Device: iPhone 15 | HN query: "iPhone 15" hate

[16/540] Device: iPhone 15 | HN query: "iPhone 15" "first impressions"
Req error: HTTPSConnectio

### Insights
- We successfully scraped `6607` rows using our initial set of devices and queries. Since this falls short of our 10,000 rows target, we are supplementing the crawl with additional device names to increase coverage. 
- The supplementary device list is composed primarily of recent or newly released models, aiming to capture newer user discussions in the HN dataset.

In [ ]:
SUPPLEMENTARY_DEVICES = [
    "iPhone 17",
    "iPhone 17 Pro",
    "iPhone 17 Pro Max",
    "Xiaomi 15 Ultra",
    "Xiaomi 15",
    "Xiaomi 14 Ultra",
    "Galaxy Z Fold5",
    "Galaxy Z Flip5",
    "Galaxy S24 Ultra",
    "Galaxy S23 Ultra",
    "Galaxy S22 Ultra",
    "Galaxy S21 Ultra",
    "OnePlus 13",
]

hn_queries_supp = build_hn_queries(SUPPLEMENTARY_DEVICES)
start = datetime(2021, 1, 1, tzinfo=timezone.utc)
end = datetime.now(timezone.utc)

df_hn_supp = crawl_hn_time_sliced(
    query_pairs=hn_queries_supp,
    start_date=start,
    end_date=end,
    slice_days=14,
    hits_per_page=500,
    max_pages_per_slice=10,
    tags="(comment,story)",
    min_chars=60,
    checkpoint_path=None,
    resume=False,
    apply_device_filter=False
)


[1/320] Device: iPhone 17 | HN query: "iPhone 17"

[2/320] Device: iPhone 17 | HN query: "iPhone 17" review

[3/320] Device: iPhone 17 | HN query: "iPhone 17" "long term review"

[4/320] Device: iPhone 17 | HN query: "iPhone 17" "worth it"

[5/320] Device: iPhone 17 | HN query: "iPhone 17" "should i buy"

[6/320] Device: iPhone 17 | HN query: "iPhone 17" upgrade

[7/320] Device: iPhone 17 | HN query: "iPhone 17" problems

[8/320] Device: iPhone 17 | HN query: "iPhone 17" issues

[9/320] Device: iPhone 17 | HN query: "iPhone 17" battery

[10/320] Device: iPhone 17 | HN query: "iPhone 17" camera

[11/320] Device: iPhone 17 | HN query: "iPhone 17" overheating

[12/320] Device: iPhone 17 | HN query: "iPhone 17" performance

[13/320] Device: iPhone 17 | HN query: "iPhone 17" regret

[14/320] Device: iPhone 17 | HN query: "iPhone 17" love

[15/320] Device: iPhone 17 | HN query: "iPhone 17" hate

[16/320] Device: iPhone 17 | HN query: "iPhone 17" "first impressions"

[17/320] Device: iPhone 

In [ ]:
df_merged = pd.concat([df_hn, df_hn_supp], ignore_index=True)

### Data crawling from official HN Firebase API

In [ ]:
df_merged = enrich_with_firebase(df_merged, wait=globals().get("FIREBASE_SLEEP", 0.15))

n_rows = len(df_merged)
n_words = int(df_merged["text"].map(word_count).sum())
n_deleted = df_merged["deleted"].sum()
print(f"Rows: {n_rows:,} | Words: {n_words:,} | Deleted: {n_deleted:,}")

  [1/10538]
  Firebase error: HTTPSConnectionPool(host='hacker-news.firebaseio.com', port=443): Max retries exceeded with url: /v0/item/37488026.json (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1010)'))) | retry in 2.7s
  Firebase error: HTTPSConnectionPool(host='hacker-news.firebaseio.com', port=443): Max retries exceeded with url: /v0/item/37607092.json (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1010)'))) | retry in 2.6s
  [500/10538]
  Firebase error: HTTPSConnectionPool(host='hacker-news.firebaseio.com', port=443): Max retries exceeded with url: /v0/item/38237194.json (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1010)'))) | retry in 1.8s
  Firebase error: HTTPSConnectionPool(host='hacker-news.firebaseio.com', port=443): Max retries exceeded with url: /v0/item/3822

## 4. Data exploration

In [ ]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10538 entries, 0 to 10537
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype              
---  ------       --------------  -----              
 0   doc_id       10538 non-null  string             
 1   object_id    10538 non-null  string             
 2   device       10538 non-null  object             
 3   title        10538 non-null  object             
 4   query        10538 non-null  object             
 5   created_at   10538 non-null  datetime64[ns, UTC]
 6   author       10538 non-null  object             
 7   tags         10538 non-null  object             
 8   source_url   10363 non-null  object             
 9   text         10538 non-null  object             
 10  deleted      10538 non-null  bool               
 11  descendants  10538 non-null  int64              
 12  score        10538 non-null  int64              
 13  item_type    10538 non-null  object             
dtypes: bool(1), datetime64

In [11]:
# Use one query and a narrow date range to get a small sample
sample_query = hn_queries[0] if "hn_queries" in globals() and hn_queries else "iPhone 15"
start_ts = int(datetime(2024, 1, 1, tzinfo=timezone.utc).timestamp())
end_ts = int(datetime(2024, 1, 31, tzinfo=timezone.utc).timestamp())

params = {
    "query": sample_query,
    "tags": "(comment,story)",
    "numericFilters": f"created_at_i>{start_ts},created_at_i<{end_ts}",
    "hitsPerPage": 2,
    "page": 0
}

print("ALGOLIA API: search_by_date")
print(f"Query: {sample_query}")
data = algolia_get(ALGOLIA_SEARCH_BY_DATE, params=params)
if data:
    print(f"\nTotal results found: {data.get('nbHits', 'N/A')} | Current page: {data.get('page', 0) + 1} | Total pages: {data.get('nbPages', 0)}")
    hits = data.get("hits", []) or []
    if hits:
        print(json.dumps(hits[0], indent=2, default=str))
    else:
        print("(no hits)")
else:
    print("(fetch failed)")

ALGOLIA API: search_by_date
Query: iPhone 15

Total results found: 116 | Current page: 1 | Total pages: 58
{
  "_highlightResult": {
    "author": {
      "matchLevel": "none",
      "matchedWords": [],
      "value": "kypro"
    },
    "comment_text": {
      "fullyHighlighted": false,
      "matchLevel": "full",
      "matchedWords": [
        "iphone",
        "15"
      ],
      "value": "I think people are overly attributing the end of low interest rates to the current job market in tech.<p>The truth is over last 10-20 years we've gone through a huge technological boom which has driven the job market in tech. In the mid 2000s almost no one knew how to code. Software engineering was more a hobby than a career choice back then. Yet, from 2005-2015 we had innovation after innovation which demanded people with coding skills.<p>This article mentions things like the launch of the <em>iPhone</em> and AWS, but there was so much more than just that. The switch from dial-up to broadband mea

In [ ]:
# Print raw Firebase API output to see what the official HN API returns
# Get one comment and one story from the corpus
story_row = df_merged[df_merged["item_type"] == "story"].head(1)
comment_row = df_merged[df_merged["item_type"] == "comment"].head(1)
story_id = story_row["object_id"].iloc[0]
comment_id = comment_row["object_id"].iloc[0]

print("FIREBASE API: STORY (id={})".format(story_id))
story = hn_firebase_get(story_id)
if story:
    print(json.dumps(story, indent=2, default=str))
else:
    print("(fetch failed)")

print("\nFIREBASE API: COMMENT (id={})".format(comment_id))
comment = hn_firebase_get(comment_id)
if comment:
    print(json.dumps(comment, indent=2, default=str))
else:
    print("(fetch failed)")

FIREBASE API: STORY (id=29632072)
{
  "by": "deepfriedginger",
  "descendants": 1,
  "id": 29632072,
  "kids": [
    29632171
  ],
  "score": 1,
  "time": 1640044464,
  "title": "Kuo: iPhone 14 to feature 48MP camera, iPhone 15 will get \u2018periscope\u2019 lens",
  "type": "story",
  "url": "https://9to5mac.com/2021/12/20/kuo-iphone-14-to-feature-48mp-lens-iphone-15-will-get-periscope-lens/"
}

FIREBASE API: COMMENT (id=25699626)
{
  "by": "pmiller2",
  "id": 25699626,
  "kids": [
    25722857
  ],
  "parent": 25699078,
  "text": "No matter how you slice it, it all eventually comes down to energy.<p>We know <i>how</i> to generate power with minimal CO2 emissions, but we don&#x27;t <i>do</i> it at the necessary scale.  We know <i>how</i> to recycle, but we don&#x27;t actually do that, either, and it takes energy. And, capturing carbon... well, if you want to put the carbon genie back in the bottle, you have to either expend energy to do it, or violate the laws of thermodynamics.  One 

In [ ]:
# Check if mapping between Algolia and Firebase is correct
idx = 0
row = df_merged.iloc[idx]
oid = row["object_id"]
ag_text = row["text"][:200] if row["text"] else ""
ag_author = row["author"]
ag_item_type = row["item_type"]

fb = hn_firebase_get(oid)

if fb:
    print("ALGOLIA AND FIREBASE MAPPING CHECK\n")
    print("object_id:", oid)
    print("\nAlgolia API:")
    print("author:", ag_author, "  item_type:", ag_item_type)
    print("text:", ag_text[:150] + "..." if len(ag_text) > 150 else ag_text)
    print("\nFirebase API:")
    print("by:", fb.get("by"), "  type:", fb.get("type"))
    fb_text = (fb.get("text") or "")[:200]
    print("text: ", fb_text[:150] + "..." if len(fb_text) > 150 else fb_text)
    print("\nAuthor/type match:", ag_author == fb.get("by") and ag_item_type == str(fb.get("type", "")))
    print("ID match:", str(oid) == str(fb.get("id", "")))
else:
    print("Firebase fetch failed for object_id:", oid)

ALGOLIA AND FIREBASE MAPPING CHECK

object_id: 25699626

Algolia API:
author: pmiller2   item_type: comment
text: CO2 already emitted will warm Earth beyond climate targets, study finds No matter how you slice it, it all eventually comes down to energy. We know ho...

Firebase API:
by: pmiller2   type: comment
text:  No matter how you slice it, it all eventually comes down to energy.<p>We know <i>how</i> to generate power with minimal CO2 emissions, but we don&#x27...

Author/type match: True
ID match: True


## 5. Data cleaning & export

In [ ]:
# Determine if a given text is low quality (too short or mostly URLs)
def is_low_quality_text(text, min_words=5, url_ratio=0.5):
    s = (text or "").strip()
    words = s.split()
    if len(words) < min_words:
        return True
    url_chars = sum(len(m) for m in re.findall(r"https?://\S+|www\.\S+", s))
    return url_chars / max(len(s), 1) > url_ratio

_ILLEGAL = re.compile(r"[\x00-\x08\x0B-\x0C\x0E-\x1F]")

# Remove illegal Excel control chars from strings
def clean_excel_cell(x):
    if x is None:
        return ""
    if isinstance(x, (int, float, bool)):
        return x
    s = str(x)
    return _ILLEGAL.sub("", s)

def make_excel_safe(df):
    df2 = df.copy()
    for c in df2.columns:
        if df2[c].dtype == "object":
            df2[c] = df2[c].map(clean_excel_cell)
    return df2

# Remove HN quotes from text
def remove_hn_quotes(text):
    if not text or not isinstance(text, str):
        return text
    lines = text.splitlines()
    out = []
    for line in lines:
        stripped = line.strip()
        if stripped.startswith(">"):
            stripped = stripped[1:].strip()  # remove leading >
        if stripped:
            out.append(stripped)
    return " ".join(out) if out else ""

In [ ]:
# Perform data cleaning steps
df_corpus = df_all.copy()

# 1. Text cleanup: cast to str, strip, unicode normalize, drop empty
df_corpus["text"] = df_corpus["text"].astype(str).str.strip()
df_corpus["text"] = df_corpus["text"].apply(
    # Use Unicode normalization form NFKC to convert characters to their canonical forms
    # e.g. "１ＡＢＣ" (full-width digits/letters) becomes "1ABC"
    lambda s: unicodedata.normalize("NFKC", s) if isinstance(s, str) else s
)
df_corpus = df_corpus[df_corpus["text"].str.len() > 0]
n_after_empty = len(df_corpus)

# 2. Remove HN quotes from text
df_corpus["text"] = df_corpus["text"].apply(remove_hn_quotes)

# 3. Quality filters: drop low-quality text, empty author
n_before_quality = len(df_corpus)
low_quality_mask = df_corpus["text"].apply(is_low_quality_text)
n_low_quality = low_quality_mask.sum()
# Print sample of low-quality texts
low_quality_df = df_corpus[low_quality_mask]
if len(low_quality_df) > 0:
    n_show = min(5, len(low_quality_df))
    print(f"Sample of low-quality texts (total filtered: {n_low_quality}):\n")
    for idx, row in low_quality_df.head(n_show).iterrows():
        t = str(row["text"] or "")
        words = len(t.split())
        url_chars = sum(len(m) for m in re.findall(r"https?://\S+|www\.\S+", t))
        ratio = url_chars / max(len(t), 1)
        preview = (t[:120] + "...") if len(t) > 120 else t
        print(f"[{words} words, url_ratio={ratio:.2f}] {preview}\n")
else:
    print("No low-quality texts found.")
df_corpus = df_corpus[~low_quality_mask]

if "author" in df_corpus.columns:
    n_before_author = len(df_corpus)
    df_corpus = df_corpus[df_corpus["author"].astype(str).str.strip().str.len() > 0]
    n_empty_author = n_before_author - len(df_corpus)
else:
    n_empty_author = 0

if "deleted" in df_corpus.columns:
    n_before_del = len(df_corpus)
    df_corpus = df_corpus[~df_corpus["deleted"].fillna(False).astype(bool)]
    n_deleted = n_before_del - len(df_corpus)
else:
    n_deleted = 0

print(f"Low quality: {n_low_quality} | Empty author: {n_empty_author} | Deleted: {n_deleted} | Rows: {n_before_quality} -> {len(df_corpus)}")

# 4. Deduplicate: one row per doc_id, then one per unique text
n_before_dedup = len(df_corpus)
if "doc_id" in df_corpus.columns:
    df_corpus = df_corpus.drop_duplicates(subset=["doc_id"])
    n_after_doc_id = len(df_corpus)
    df_corpus = df_corpus.drop_duplicates(subset=["text"]).reset_index(drop=True)
    n_after_text_dedup = len(df_corpus)
    print(f"Doc_id duplicates removed: {n_before_dedup - n_after_doc_id} | Text duplicates removed: {n_after_doc_id - n_after_text_dedup} | Rows: {n_before_dedup} -> {n_after_text_dedup}")
else:
    df_corpus = df_corpus.drop_duplicates(subset=["text"]).reset_index(drop=True)
    print(f"Text duplicates removed: {n_before_dedup - len(df_corpus)} | Rows: {n_before_dedup} -> {len(df_corpus)}")

# 5. Remove illegal Excel chars before export
df_corpus = make_excel_safe(df_corpus)

Sample of low-quality texts (total filtered: 11):

[25 words, url_ratio=0.57] iPhone 14 Pro camera review: A small step, a huge leap Not my gallery, but a good side by side: https://imgur.io/a/vG6Ys...

[17 words, url_ratio=0.51] US smartphone shipments fall sharply, but Android more than iPhone Unlocked direct from the manufacturer: https://www.ap...

[16 words, url_ratio=0.58] Modern smartphone lenses are crazy Of course you can. Stars example: https://www.macrumors.com/2021/10/10/amazing-night-...

[12 words, url_ratio=0.51] iPhone Air It's 17% heavier than the iphone 13 mini. Source: https://www.apple.com/iphone/compare/?modelList=iphone-13-m...

[7 words, url_ratio=0.64] iPhone Pocket https://irepart.com/products/iphone-13-mini-battery-replacem... Original was 2406 mAh

Low quality: 11 | Empty author: 0 | Deleted: 0 | Rows: 10538 -> 10527
Doc_id duplicates removed: 0 | Text duplicates removed: 0 | Rows: 10527 -> 10527


In [ ]:
df_corpus_excel = df_corpus.copy()

# Remove timezone information from datetime columns if any
for col in df_corpus_excel.columns:
    if "datetime" in str(df_corpus_excel[col].dtype).lower():
        if hasattr(df_corpus_excel[col].dt, "tz_localize"):
            df_corpus_excel[col] = df_corpus_excel[col].dt.tz_localize(None)

# Save outputs
df_corpus_excel.to_excel("data/raw-data/corpus_full.xlsx", index=False, engine="openpyxl")
print("Saved corpus_full.xlsx | rows:", len(df_corpus_excel), "| cols:", len(df_corpus_excel.columns))

df_eval = df_corpus_excel[["text"]].copy()
df_eval["label"] = ""
df_eval.to_excel("data/raw-data/eval.xlsx", index=False, engine="openpyxl")
print("Saved eval.xlsx | rows:", len(df_eval))

Saved corpus_full.xlsx | rows: 10527 | cols: 14
Saved eval.xlsx | rows: 10527
